# Research-grade expansion: GM12878 autosomes and K562

This notebook has two distinct purposes:

1. **Reproduce the already validated result** on held-out GM12878
   chromosome 21.
2. **Run new confirmatory experiments** on previously unused GM12878
   chromosomes and matched K562 data.

The large-data switches are off by default. Turning them on downloads
approximately 2.3 GB of ATAC bigWigs and reads large remote Hi-C files.
Do not report expansion results until those cells finish and their
outputs have been reviewed.

## 1. Setup

In Colab, upload and unzip the project first, then change into its root
directory. On a local Jupyter server, start the notebook from the
repository root.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import Image, display

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/content/ATAC_HiC_research_grade"),
]
PROJECT_ROOT = next(
    (path.resolve() for path in candidates if (path / "src").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the project root. Unzip the project and "
        "change into the directory containing src/, scripts/, and data/."
    )

sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.environ["MPLCONFIGDIR"] = str(PROJECT_ROOT / ".matplotlib")
print("Project root:", PROJECT_ROOT)

In [ ]:
# Run this installation once in a fresh Colab runtime.
# It installs the exact package versions used by the project.
INSTALL_DEPENDENCIES = False

if INSTALL_DEPENDENCIES:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-e",
            str(PROJECT_ROOT),
        ],
        check=True,
    )
    print("Dependencies installed. Restart the runtime if Colab asks.")
else:
    print("Dependency installation skipped.")

## 2. Reproduce the completed held-out result

This uses the included processed chr16–21 dataset. It trains on
chr16–19, selects settings on chr20, and evaluates the long arm of
chr21. Expected MSE improvement is approximately **29.57%**.

In [ ]:
from atac_hic.data import validate_processed_dataset

pilot_data_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "GM12878_VC_SQRT_chr16_21.npz"
)
with np.load(pilot_data_path, allow_pickle=False) as pilot_data:
    validation_table = pd.DataFrame(
        validate_processed_dataset(
            pilot_data,
            [f"chr{x}" for x in range(16, 22)],
        )
    )
validation_table

In [ ]:
reproduction_directory = (
    PROJECT_ROOT / "results" / "notebook_reproduction"
)
subprocess.run(
    [
        sys.executable,
        str(PROJECT_ROOT / "scripts" / "run_validated.py"),
        "--data",
        str(pilot_data_path),
        "--output-directory",
        str(reproduction_directory),
    ],
    check=True,
    env={
        **os.environ,
        "PYTHONPATH": str(PROJECT_ROOT / "src"),
    },
)

with open(reproduction_directory / "primary_results.json") as handle:
    primary_results = json.load(handle)
pd.Series(primary_results, name="value").to_frame()

In [ ]:
display(
    Image(
        filename=str(
            reproduction_directory
            / "chr21_validation_diagnostics.png"
        )
    )
)

## 3. Confirm the dataset manifest

These are matched GRCh38 ENCODE output types. The accessions and MD5
values are fixed before the expansion is run.

In [ ]:
with open(PROJECT_ROOT / "configs" / "datasets.json") as handle:
    dataset_manifest = json.load(handle)
pd.DataFrame(dataset_manifest).T

## 4. Extract full-autosome datasets

Set the GM12878 switch first. Run and inspect its confirmatory results
before adding K562. Extraction can take tens of minutes depending on
ENCODE and Colab network speed.

In [ ]:
RUN_GM12878_EXTRACTION = False
RUN_K562_EXTRACTION = False

processed_directory = PROJECT_ROOT / "data" / "processed"
processed_directory.mkdir(parents=True, exist_ok=True)
gm_autosomes = (
    processed_directory / "GM12878_VC_SQRT_autosomes.npz"
)
k562_autosomes = (
    processed_directory / "K562_VC_SQRT_autosomes.npz"
)

def extract(dataset_name, destination):
    subprocess.run(
        [
            sys.executable,
            str(PROJECT_ROOT / "scripts" / "extract_encode.py"),
            "--dataset",
            dataset_name,
            "--output",
            str(destination),
        ],
        check=True,
        env={
            **os.environ,
            "PYTHONPATH": str(PROJECT_ROOT / "src"),
        },
    )

if RUN_GM12878_EXTRACTION:
    extract("gm12878", gm_autosomes)
else:
    print("GM12878 extraction is off.")

if RUN_K562_EXTRACTION:
    extract("k562", k562_autosomes)
else:
    print("K562 extraction is off.")

## 5. Locked-model confirmation and external validation

The GM12878 confirmatory chromosomes are chr1–15 and chr22. They are
untouched by the original pilot. If a K562 dataset is present, the same
command also runs the pre-specified within-K562 split and exploratory
zero-shot transfer.

In [ ]:
RUN_EXPANSION = False
expansion_directory = PROJECT_ROOT / "results" / "expansion"

if RUN_EXPANSION:
    if not gm_autosomes.exists():
        raise FileNotFoundError(
            "Extract the GM12878 autosome dataset first."
        )
    command = [
        sys.executable,
        str(PROJECT_ROOT / "scripts" / "run_expansion.py"),
        "--gm12878-data",
        str(gm_autosomes),
        "--output-directory",
        str(expansion_directory),
    ]
    if k562_autosomes.exists():
        command.extend(["--k562-data", str(k562_autosomes)])
    subprocess.run(
        command,
        check=True,
        env={
            **os.environ,
            "PYTHONPATH": str(PROJECT_ROOT / "src"),
        },
    )
else:
    print("Expansion is off; no unvalidated claims were generated.")

In [ ]:
summary_path = expansion_directory / "expansion_summary.json"
if summary_path.exists():
    with open(summary_path) as handle:
        expansion_summary = json.load(handle)
    display(pd.Series(expansion_summary, name="value").to_frame())

    for filename in [
        "GM12878_confirmatory_chromosomes.csv",
        "K562_within_cell_validation.csv",
        "GM12878_to_K562_transfer.csv",
    ]:
        path = expansion_directory / filename
        if path.exists():
            print("\n", filename)
            display(pd.read_csv(path))
else:
    print("No expansion result exists yet.")

## 6. Interpretation rules

- The primary comparison is always against the distance-only baseline.
- Report every held-out chromosome, including failures.
- Use the chromosome-bootstrap interval, not pair-level p-values.
- Describe within-K562 validation separately from GM12878-to-K562
  transfer.
- A successful result is predictive association, not causality and not
  full Hi-C reconstruction from ATAC alone.
- Update `docs/VALIDATED_REPORT.md` only after the new outputs have been
  reviewed.